In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")


In [0]:
required_tables = [
    "company_ro.gold.fact_financiar_anual",
    "company_ro.gold.dim_firma",
    "company_ro.gold.dim_stare_firma",
    "company_ro.gold.dim_forma_juridica",
    "company_ro.gold.dim_localitate",
    "company_ro.gold.dim_adresa",
    "company_ro.gold.dim_caen",
]

for table_name in required_tables:
    print(table_name, spark.table(table_name).count())


In [0]:
company_financial_summary = spark.sql("""
WITH fact_normalized AS (
  SELECT
    *,
    LPAD(
      REGEXP_REPLACE(CAST(cod_caen_mfinante AS STRING), '[^0-9]', ''),
      4,
      '0'
    ) AS cod_caen_mfinante_4
  FROM company_ro.gold.fact_financiar_anual
)

SELECT
  f.an,
  f.cui,

  d.denumire,
  d.cod_inmatriculare,
  l.judet,
  l.localitate,
  a.adresa,
  d.cod_stare_firma,
  s.stare_firma,
  s.is_activa,
  fj.forma_juridica,
  TRY_CAST(REGEXP_EXTRACT(d.cod_inmatriculare, '/([0-9]{4})$', 1) AS INT) AS an_infiintare,

  f.cod_caen_mfinante_4 AS cod_caen_mfinante,
  COALESCE(c.cod_caen, f.cod_caen_mfinante_4) AS cod_caen,
  c.denumire_caen,
  SUBSTRING(COALESCE(c.cod_caen, f.cod_caen_mfinante_4), 1, 3) AS grupa_caen,
  COALESCE(c.cod_caen, f.cod_caen_mfinante_4) AS clasa_caen,

  CONCAT(
    f.cod_caen_mfinante_4,
    ' - ',
    COALESCE(c.denumire_caen, 'Unknown CAEN')
  ) AS caen_display_name,

  f.cifra_afaceri,
  f.venituri_totale,
  f.cheltuieli_totale,
  f.profit_brut,
  f.pierdere_bruta,
  f.profit_net,
  f.pierdere_neta,
  f.nr_mediu_salariati,

  f.datorii,
  f.capitaluri_proprii,
  f.capital_social,
  f.active_imobilizate,
  f.active_circulante,
  f.stocuri,
  f.creante,
  f.casa_si_conturi,

  f._ingested_at,
  f._source_file

FROM fact_normalized f
LEFT JOIN company_ro.gold.dim_firma d
  ON f.cui = d.cui
LEFT JOIN company_ro.gold.dim_stare_firma s
  ON d.cod_stare_firma = s.cod_stare_firma
LEFT JOIN company_ro.gold.dim_forma_juridica fj
  ON d.forma_juridica_key = fj.forma_juridica_key
LEFT JOIN company_ro.gold.dim_adresa a
  ON d.adresa_key = a.adresa_key
LEFT JOIN company_ro.gold.dim_localitate l
  ON a.localitate_key = l.localitate_key
LEFT JOIN (
  SELECT cod_caen, denumire_caen, grupa_caen
  FROM company_ro.gold.dim_caen
  WHERE versiune_caen = '0'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY cod_caen ORDER BY _ingested_at DESC) = 1
) c
  ON f.cod_caen_mfinante_4 = c.cod_caen
""")

display(company_financial_summary.limit(50))
print("Rows:", company_financial_summary.count())
print("Columns:", company_financial_summary.columns)


In [0]:
(
    company_financial_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.gold.company_financial_summary")
)


In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows,
  COUNT(DISTINCT cui) AS companies,
  SUM(cifra_afaceri) AS total_cifra_afaceri,
  SUM(profit_net) AS total_profit_net,
  SUM(nr_mediu_salariati) AS total_salariati
FROM company_ro.gold.company_financial_summary
GROUP BY an
ORDER BY an
"""))

display(spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(denumire) AS rows_with_company_name,
  COUNT(judet) AS rows_with_judet,
  COUNT(stare_firma) AS rows_with_status,
  COUNT(forma_juridica) AS rows_with_legal_form,
  COUNT(denumire_caen) AS rows_with_caen_name,
  COUNT(cifra_afaceri) AS rows_with_revenue
FROM company_ro.gold.company_financial_summary
"""))


In [0]:
# Test the corrected an_infiintare extraction
print("=== Testing an_infiintare fix ===")
display(spark.sql("""
SELECT 
  cod_inmatriculare,
  REGEXP_EXTRACT(cod_inmatriculare, '/(\\d{4})$', 1) AS extracted_year_corrected,
  TRY_CAST(REGEXP_EXTRACT(cod_inmatriculare, '/(\\d{4})$', 1) AS INT) AS an_infiintare_corrected
FROM company_ro.gold.dim_firma
WHERE cod_inmatriculare IS NOT NULL
LIMIT 10
"""))

print("\n=== Coverage with corrected pattern ===")
display(spark.sql("""
SELECT 
  COUNT(*) AS total_companies,
  COUNT(CASE WHEN TRY_CAST(REGEXP_EXTRACT(cod_inmatriculare, '/(\\d{4})$', 1) AS INT) IS NOT NULL 
             THEN 1 END) AS with_extracted_year,
  ROUND(COUNT(CASE WHEN TRY_CAST(REGEXP_EXTRACT(cod_inmatriculare, '/(\\d{4})$', 1) AS INT) IS NOT NULL 
             THEN 1 END) * 100.0 / COUNT(*), 2) AS percent_coverage
FROM company_ro.gold.dim_firma
WHERE cod_inmatriculare IS NOT NULL
"""))

# Test venituri_totale vs cifra_afaceri coverage
print("\n=== Revenue column comparison (2020-2025) ===")
display(spark.sql("""
SELECT 
  an,
  COUNT(*) AS total_records,
  COUNT(cifra_afaceri) AS has_cifra_afaceri,
  COUNT(venituri_totale) AS has_venituri_totale,
  COUNT(COALESCE(cifra_afaceri, venituri_totale)) AS has_any_revenue,
  ROUND(COUNT(cifra_afaceri) * 100.0 / COUNT(*), 2) AS pct_cifra_afaceri,
  ROUND(COUNT(venituri_totale) * 100.0 / COUNT(*), 2) AS pct_venituri_totale,
  ROUND(COUNT(COALESCE(cifra_afaceri, venituri_totale)) * 100.0 / COUNT(*), 2) AS pct_any_revenue
FROM company_ro.gold.fact_financiar_anual
WHERE an >= 2020
GROUP BY an
ORDER BY an
"""))